In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

### OKCupid Profiles

In [ ]:
okcupid = pd.read_csv('../data/okcupid_profiles.csv')

print(okcupid.columns)
print('===========')
print(okcupid.describe())

In [ ]:
sns.histplot(okcupid['sex'])
plt.show()

In [ ]:
sns.histplot(data=okcupid, x='status', hue='sex')
plt.show()

In [ ]:
sns.histplot(data=okcupid, x='orientation', hue='sex')
plt.show()

In [ ]:
print(len(okcupid[okcupid.height.isna()]))
sns.histplot(data=okcupid, x='height', hue='sex', kde=True)
plt.show()

In [ ]:
print(len(okcupid[okcupid.drinks.isna()]))
sns.histplot(data=okcupid, x='drinks', hue='sex')
plt.xticks(rotation=10)
plt.show()

In [ ]:
print(len(okcupid[okcupid.smokes.isna()]))
sns.histplot(data=okcupid, x='smokes', hue='sex')
plt.show()

In [ ]:
print(len(okcupid[okcupid.diet.isna()]))
sns.histplot(data=okcupid, x='diet', hue='sex')
plt.xticks(rotation=80)
plt.show()

In [ ]:
cupid_text = okcupid.copy()

essay_columns = cupid_text.iloc[:,21:].columns
for col in essay_columns:
    cupid_text[col] = cupid_text[col].fillna(' ')

cupid_text["essay"] = (cupid_text["essay0"].str
                       .cat(cupid_text.iloc[:,22:].astype(str), sep=" "))

cupid_text = cupid_text.drop(cupid_text.iloc[:,21:-1], axis=1)

cupid_text["total words"] = cupid_text["essay"].str.split().str.len()

cupid_text['total words'].head()

In [ ]:
okcupid.sign.value_counts().head()

In [ ]:
okcupid[['height','income','job','last_online','location',
         'offspring','pets','religion','smokes','speaks']].head()

In [ ]:
okcupid.location.value_counts()

In [ ]:
okcupid.religion.value_counts().head()

In [ ]:
print(len(okcupid[okcupid.offspring.isna()]))
okcupid.offspring.value_counts().head()

### Separation of Columns

In [ ]:
signs = ['aries','taurus','gemini','cancer','leo','virgo','libra',
         'scorpio','sagittarius','capricorn','aquarius','pisces']

opinions = ['fun to think about', 'matters a lot', 'but']

educ_status = ['graduated from', 'working on', 'dropped out']

schools = ['college/university', 'masters', 'two-year college',
           'high school', 'ph.d', 'law school', 'space camp',
           'med school']

pet_status = ['likes dogs', 'likes cats', 'dislikes dogs',
              'dislikes cats', 'has dogs', 'has cats']

religions = ['agnosticism','islam','hinduism','judaism','atheism',
             'christianity','other','buddhism','catholicism']

religion_attitude = ['not too serious', 'laughing about it',
                     'somewhat serious', 'very serious']

ethnicities = ['white','black','asian','hispanic / latin',
               'middle eastern','indian','pacific islander',
               'native american','other']

In [ ]:
cupid_signs = cupid_text.copy()

cupid_signs.drugs = cupid_signs.drugs.fillna('never')
cupid_signs.drinks = cupid_signs.drinks.fillna('socially')
cupid_signs.smokes = cupid_signs.smokes.fillna('no')
cupid_signs.body_type = cupid_signs.body_type.fillna('average')
cupid_signs.height = cupid_signs.height.fillna(cupid_signs.height.median())
cupid_signs.offspring = cupid_signs.offspring.fillna(0)
cupid_signs.pets = cupid_signs.pets.fillna('neutral on dogs and neutral on cats')

In [ ]:
for zodiac in signs:
    cupid_signs[zodiac] = (cupid_signs.sign.str.contains(zodiac))

for opinion in opinions:
    cupid_signs[opinion] = (cupid_signs.sign.str.contains(opinion))

for educ in educ_status:
    cupid_signs[educ] = (cupid_signs.education.str.contains(educ))

for school in schools:
    cupid_signs[school] = (cupid_signs.education.str.contains(school))

for pet in pet_status:
    cupid_signs[pet] = (cupid_signs.pets.str.contains(pet))

for rel in religions:
    cupid_signs[rel] = (cupid_signs.religion.str.contains(rel))

for attitude in religion_attitude:
    cupid_signs[attitude] = (cupid_signs.religion.str.contains(attitude))

cupid_signs.rename(columns={'other':'other religion'}, inplace=True)

for eth in ethnicities:
    cupid_signs[eth] = (cupid_signs.ethnicity.str.contains(eth))
    cupid_signs[eth] = pd.to_numeric(cupid_signs[eth], downcast='integer', errors='coerce')

In [ ]:
cupid_signs.rename(columns={'but':'zodiac does not matter',
                            'matters a lot':'zodiac matters a lot',
                            'fun to think about':'zodiac fun to think about',
                            'not too serious':'religion not too serious',
                            'laughing about it':'laughing about religion',
                            'somewhat serious':'religion somewhat serious',
                            'very serious':'religion very serious',
                            'other':'other ethnicity'}, inplace=True)

split_location = cupid_signs.location.str.split(', ', n=1, expand=True)

cupid_signs['city'] = split_location[0]
cupid_signs['state'] = split_location[1]

cupid_signs.drop(columns=['location','education','religion',
                          'sign','pets','ethnicity'], inplace=True)

cupid_signs.head()

In [ ]:
print('Asian:',cupid_signs['asian'].sum(),
      'White:',cupid_signs['white'].sum(),
      'Black:',cupid_signs['black'].sum(),
      'Hispanic/Latin:',cupid_signs['hispanic / latin'].sum(),
      'Indian:',cupid_signs['indian'].sum(),
      'Middle Eastern:',cupid_signs['middle eastern'].sum(),
      'Pacific Islander:',cupid_signs['pacific islander'].sum(),
      'Native American:',cupid_signs['native american'].sum(),
      'Other:',cupid_signs['other ethnicity'].sum())

In [ ]:
cupid_signs.state.value_counts().head()

In [ ]:
cupid_signs.iloc[:, 17:71].sum().head()

In [ ]:
missing_data = cupid_signs.isnull().sum()
print(missing_data[missing_data > 0])

In [ ]:
sns.histplot(data=cupid_signs, x='age', hue='sex', kde=True)
plt.title('Distribution of Age')
plt.show()

### Text Vectorization

In [ ]:
def lemmatize_sentence(sentence):
    lemmatizer = WordNetLemmatizer()
    tokens = word_tokenize(sentence)
    lemmatized_tokens = [lemmatizer.lemmatize(token)+' ' for token in tokens]
    return ''.join(lemmatized_tokens)

In [ ]:
cupid_signs['essayLemmatized'] = cupid_signs['essay'].apply(lemmatize_sentence)

In [ ]:
min_ngram = 1
max_ngram = 3
stop_words = 'english'
vectorizer = 15

okcupid_essays = cupid_signs

tfidf_vectorizer = TfidfVectorizer(
    max_features=vectorizer,
    stop_words=stop_words,
    ngram_range=(min_ngram,max_ngram)
    )


In [ ]:
tfidf_model = tfidf_vectorizer.fit_transform(okcupid_essays['essayLemmatized'])
tfidf_text = tfidf_vectorizer.get_feature_names_out()
tfidf_stop_words_vect = tfidf_vectorizer.get_stop_words()
tfidf_columns_text = ["containsTfidf: "+ item for item in tfidf_text]

tfidf_words_df = pd.DataFrame(
    tfidf_model.toarray(),
    columns=tfidf_text)

essay_final = okcupid_essays.join(tfidf_words_df).fillna('')

In [ ]:
count_vectorizer = CountVectorizer(
    max_features=vectorizer,
    stop_words=stop_words,
    ngram_range=(min_ngram,max_ngram)
    )

count_model = count_vectorizer.fit_transform(okcupid_essays['essayLemmatized'])
count_text = count_vectorizer.get_feature_names_out()
count_stop_words_vect = count_vectorizer.get_stop_words()
count_columns_text = ["containsCount: "+ item for item in count_text]

count_words_df = pd.DataFrame(
    count_model.toarray(),
    columns=count_columns_text)

essay_final = essay_final.join(count_words_df).fillna('')

In [ ]:
tfidf_words_df.head()

In [ ]:
count_words_df.head()

In [ ]:
features = ['age', 'height', 'book', 'food', 'friend',
            'movie', 'music', 'life',
            'new', 'people',
            'thing', 'time',
            'containsCount: book', 'containsCount: food',
            'containsCount: friend',
            'containsCount: life',
            'containsCount: movie', 'containsCount: music',
            'containsCount: new', 'containsCount: people',
            'containsCount: thing', 'containsCount: time',
            'sex', 'orientation', 'body_type',
            'smokes', 'drinks', 'drugs', 
            'graduated from', 'working on', 'dropped out',
            'college/university', 'masters', 'two-year college',
            'high school', 'ph.d', 'law school', 'space camp',
            'med school',
            'agnosticism',
            'islam', 'hinduism', 'judaism', 'atheism', 'christianity',
            'other religion', 'buddhism', 'catholicism',
            'religion not too serious', 'laughing about religion',
            'religion somewhat serious', 'religion very serious',
            'white', 'black', 'asian', 'hispanic / latin',
            'middle eastern', 'indian', 'pacific islander',
            'native american', 'other ethnicity']

X = essay_final[features[0:22]].copy()
X.head()

In [ ]:
for feat in features[22:]:
    X[feat] = pd.Categorical(cupid_signs[feat])
    X[feat] = X[feat].cat.codes

scaler = StandardScaler()
scaler.set_output(transform='pandas')

X_scaled = scaler.fit_transform(X)

In [ ]:
num_clusters=list(np.arange(1, 21)) # bigger numbers higher chance of crash

inertias = []

for k in num_clusters:
  model = KMeans(n_clusters = k, n_init='auto')
  model.fit(X_scaled)
  inertias.append(model.inertia_)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(num_clusters, inertias, '-o')
ax.set(xticks=num_clusters, xlabel='clusters', title='Inertia')
plt.show()

In [ ]:
k = 19
model = KMeans(n_clusters = k, n_init='auto', random_state=42)
model.fit(X_scaled)
X['cluster'] = model.labels_
X.cluster.value_counts()

fig2, ax2 = plt.subplots(figsize=(14, 10))

sns.countplot(data=X, x='cluster', ax=ax2, hue='sex')
ax2.set(title = 'Users per cluster', ylabel='')
plt.show()

In [ ]:
tsne_cluster = TSNE(n_components=2, random_state=42)
tsne_cluster_result = tsne_cluster.fit_transform(X) # very long runtime

tsne_cluster_result

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(x=tsne_cluster_result[:, 0], y=tsne_cluster_result[:, 1], hue=X['cluster'])

plt.title('t-SNE of OKCupid Clusters')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(x=tsne_cluster_result[:, 0], y=tsne_cluster_result[:, 1], hue=X['sex'])

plt.title('t-SNE of OKCupid Clusters colored by Sex')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.show()

In [ ]:
X_nonwhite = X[X.white == 0]

tsne_cluster_result_nonwhite = tsne_cluster.fit_transform(X_nonwhite)

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(x=tsne_cluster_result_nonwhite[:, 0], y=tsne_cluster_result_nonwhite[:, 1], hue=X_nonwhite['cluster'])

plt.title('t-SNE of OKCupid Clusters (nonwhite users)')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(x=tsne_cluster_result_nonwhite[:, 0], y=tsne_cluster_result_nonwhite[:, 1], hue=X_nonwhite['sex'])

plt.title('t-SNE of OKCupid Clusters (nonwhite users), colored by Sex')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.show()

In [ ]:
X_white = X[X.white == 1]

tsne_cluster_result_white = tsne_cluster.fit_transform(X_white)

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(x=tsne_cluster_result_white[:, 0], y=tsne_cluster_result_white[:, 1], hue=X_white['cluster'])

plt.title('t-SNE of OKCupid Clusters (white users)')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(x=tsne_cluster_result_white[:, 0], y=tsne_cluster_result_white[:, 1], hue=X_white['sex'])

plt.title('t-SNE of OKCupid Clusters (white users), colored by Sex')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.show()

In [ ]:
X[X.cluster == 0].describe()

In [ ]:
X_white.cluster.value_counts()

In [ ]:
X_nonwhite.cluster.value_counts()

In [ ]:
X[X.asian == 1].cluster.value_counts()

In [ ]:
X[X.black == 1].cluster.value_counts()

In [ ]:
X[X['hispanic / latin'] == 1].cluster.value_counts()

In [ ]:
X[X['middle eastern'] == 1].cluster.value_counts()

In [ ]:
X[X['other ethnicity'] == 1].cluster.value_counts()

In [ ]:
essay_sum = pd.DataFrame(tfidf_words_df.sum(), columns=['sum'])
essay_sum.reset_index(inplace=True)
essay_sum.rename(columns={'index':'word'}, inplace=True)

top50_essay = essay_sum.sort_values(by='sum', ascending=False)[:50]

top50_essay